In [2]:
import argparse
import os
import sys
from rsa_utils import (
    create_association_model_rdm,
    create_identity_model_rdm,
    searchlight_rsa_single_subject,
    save_rsa_maps
)

In [3]:
association_rdm = create_association_model_rdm()
identity_rdm = create_identity_model_rdm()
    
model_rdms = {
        'association': association_rdm,
        'identity': identity_rdm
}

In [4]:
association_rdm

array([[1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1.],
       [0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 1.]])

In [5]:
identity_rdm

array([[1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 1.],
       [1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 1.],
       [1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 1.],
       [1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0.],
       [0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0., 1., 0., 0., 1., 0., 0., 1.]])

In [7]:
import numpy as np
from scipy.stats import pearsonr, spearmanr

# number of conditions
n = association_rdm.shape[0]

# lower triangle indices, excluding diagonal
tril_idx = np.tril_indices(n, k=-1)

# vectorize
v_assoc = association_rdm[tril_idx]
v_ident = identity_rdm[tril_idx]

# Pearson correlation
r_pearson, p_pearson = pearsonr(v_assoc, v_ident)

# Spearman correlation (often preferred for binary RDMs)
r_spearman, p_spearman = spearmanr(v_assoc, v_ident)

print("Pearson r:", r_pearson, "p =", p_pearson)
print("Spearman r:", r_spearman, "p =", p_spearman)


Pearson r: 0.5163977794943228 p = 9.062735972690181e-06
Spearman r: 0.5163977794943224 p = 9.062735972690329e-06


In [5]:
sub='sub-01'
base_dir='/Users/liuq13/SPCD_data'
session_folder='premeal_trialwise_2mm'
session_tag='premeal'
mask_file=os.path.join(base_dir, "masks", "gm_0.1.nii")
radisu=8
n_jobs=-1


In [ ]:
beta_maps = searchlight_rsa_single_subject(
            sub=sub,
            base_dir=base_dir,
            session_folder=session_folder,
            session_tag=session,
            mask_file=mask,
            model_rdms=model_rdms,
            radius=radius,
            n_jobs=n_jobs
        )

In [10]:

import os
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from sklearn.linear_model import LinearRegression
from nilearn.image import load_img, concat_imgs, get_data, new_img_like

# ============================================================================
# Data Loading Functions
# ============================================================================

def _subject_run_dir(base_dir: str, sub: str, session_folder: str):
    """Return the subject/session folder that holds CSVs and beta NIfTIs."""
    return os.path.join(base_dir, "GLM", session_folder, sub)


def _csv_path_for_run(sub_dir: str, sub: str, run: int, session: str):
    """Return path to beta_index CSV for a given run."""
    return os.path.join(sub_dir, f"{sub}_{session}_run-{run}_beta_index.csv")


def load_run_betas_rsa(sub: str,
                        run: int,
                        base_dir: str,
                        session_folder: str,
                        session_tag: str):
    """
    Load all trial-wise betas for one run, with z-score normalization.
    
    Parameters
    ----------
    sub : str
        Subject ID (e.g., 'sub-01')
    run : int
        Run number (1, 2, or 3)
    base_dir : str
        Base directory containing GLM folder
    session_folder : str
        Folder name (e.g., 'premeal_trialwise_2mm')
    session_tag : str
        Session tag for CSV naming (e.g., 'premeal')
    
    Returns
    -------
    imgs : list of NIfTI images (z-scored)
    metadata : pd.DataFrame with columns ['run', 'set', 'cue_index', 'odor_index', 'beta_path']
    """
    sub_dir = _subject_run_dir(base_dir, sub, session_folder)
    csv_path = _csv_path_for_run(sub_dir, sub, run, session_tag)
    
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}")
    
    # Load metadata
    df = pd.read_csv(csv_path)
    df["beta_path"] = df["beta_file"].apply(lambda b: os.path.join(sub_dir, b))
    
    # Load all betas for this run
    beta_paths = df["beta_path"].tolist()
    ref_img = None
    stack = []
    
    for p in beta_paths:
        img = load_img(p)
        if ref_img is None:
            ref_img = img
        stack.append(get_data(img))
    
    # Stack and z-score across ALL trials in this run
    all_stack = np.stack(stack, axis=0)  # (n_trials, X, Y, Z)
    mean = all_stack.mean(axis=0)
    std = all_stack.std(axis=0)
    std[std == 0] = 1.0  # Avoid division by zero
    
    # Normalize each trial
    norm_imgs = []
    for data in all_stack:
        z = (data - mean) / std
        norm_imgs.append(new_img_like(ref_img, z))
    
    # Metadata
    metadata = df.loc[:, ["run", "set", "cue_index", "odor_index"]].copy()
    metadata["beta_path"] = beta_paths
    
    return norm_imgs, metadata

In [11]:
condition_map = { # very useful to map "meta" data to condition info
        (1, 1): 'A11',  # Set1, A, odor1
        (3, 2): 'A21',  # Set1, A, odor2
        (5, 0): 'C1',   # Set1, A, clean
        (2, 1): 'B11',  # Set1, B, odor1
        (4, 2): 'B21',  # Set1, B, odor2
        (6, 0): 'D1',   # Set1, B, clean
        (7, 1): 'A12',  # Set2, A, odor1
        (9, 2): 'A22',  # Set2, A, odor2
        (11, 0): 'C2',  # Set2, A, clean
        (8, 1): 'B12',  # Set2, B, odor1
        (10, 2): 'B22', # Set2, B, odor2
        (12, 0): 'D2',  # Set2, B, clean
    }

In [13]:
condition_trials = {cond: [] for cond in condition_map.values()}
condition_trials

{'A11': [],
 'A21': [],
 'C1': [],
 'B11': [],
 'B21': [],
 'D1': [],
 'A12': [],
 'A22': [],
 'C2': [],
 'B12': [],
 'B22': [],
 'D2': []}

In [15]:
sub='sub-01'
runs = (1,2,3)
base_dir='/Users/liuq13/SPCD_data'
session_folder='premeal_trialwise_2mm'
session_tag='premeal'

In [16]:
for run in runs:
        try:
            imgs, meta = load_run_betas_rsa(
                sub, run, base_dir, session_folder, session_tag
            )
        except FileNotFoundError:
            print(f"Warning: Run {run} not found for {sub}, skipping")
            continue
        
        for idx, row in meta.iterrows():
            key = (int(row['cue_index']), int(row['odor_index']))
            if key in condition_map:
                cond_name = condition_map[key]
                condition_trials[cond_name].append(imgs[idx]) 
                # 4 images per condition (if not missing data)

In [17]:
condition_trials

{'A11': [<nibabel.nifti1.Nifti1Image at 0x1124de460>,
 'A21': [<nibabel.nifti1.Nifti1Image at 0x112484610>,
 'C1': [<nibabel.nifti1.Nifti1Image at 0x117ab1520>,
 'B11': [<nibabel.nifti1.Nifti1Image at 0x16a1ca610>,
 'B21': [<nibabel.nifti1.Nifti1Image at 0x16a256d60>,
 'D1': [<nibabel.nifti1.Nifti1Image at 0x16a1ca160>,
 'A12': [<nibabel.nifti1.Nifti1Image at 0x16a1caca0>,
 'A22': [<nibabel.nifti1.Nifti1Image at 0x16a1ca550>,
 'C2': [<nibabel.nifti1.Nifti1Image at 0x16a1ca190>,
 'B12': [<nibabel.nifti1.Nifti1Image at 0x16a1caeb0>,
 'B22': [<nibabel.nifti1.Nifti1Image at 0x16a249460>,
 'D2': [<nibabel.nifti1.Nifti1Image at 0x16a1cab80>,
  <nibabel.nifti1.Nifti1Image at 0x16a248c10>]}

In [18]:
condition_imgs = {}

In [19]:
for cond_name, trial_imgs in condition_trials.items():
        if len(trial_imgs) == 0:
            raise ValueError(f"No trials found for condition {cond_name}")
        
        # Stack and average
        trial_data = np.stack([get_data(img) for img in trial_imgs], axis=0)
        avg_data = trial_data.mean(axis=0)
        condition_imgs[cond_name] = new_img_like(trial_imgs[0], avg_data)

In [20]:
condition_imgs

{'A11': <nibabel.nifti1.Nifti1Image at 0x147629c10>,
 'A21': <nibabel.nifti1.Nifti1Image at 0x16a253550>,
 'C1': <nibabel.nifti1.Nifti1Image at 0x16a1b9f10>,
 'B11': <nibabel.nifti1.Nifti1Image at 0x16a1bca00>,
 'B21': <nibabel.nifti1.Nifti1Image at 0x16a1bcbe0>,
 'D1': <nibabel.nifti1.Nifti1Image at 0x16a1bc7f0>,
 'A12': <nibabel.nifti1.Nifti1Image at 0x16a24c310>,
 'A22': <nibabel.nifti1.Nifti1Image at 0x16a24cc70>,
 'C2': <nibabel.nifti1.Nifti1Image at 0x16a24cbe0>,
 'B12': <nibabel.nifti1.Nifti1Image at 0x16a24cb20>,
 'B22': <nibabel.nifti1.Nifti1Image at 0x16a24cd30>,
 'D2': <nibabel.nifti1.Nifti1Image at 0x16a24cee0>}